In [ ]:
!pip install -U transformers==4.57.1 peft==0.20.0 datasets==4.3.0 accelerate==1.11.0 bitsandbytes==0.48.2 trl==0.23.0

In [ ]:
import sys
import torch
import transformers
import peft
import datasets
import accelerate
import bitsandbytes
import trl

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

In [ ]:
import os

dataset_path = "/kaggle/input/datasets/shouldichangemyname/dcot-traindataset/dcot_trainable.json"

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=dataset_path
)

print(dataset)
print("Training shape:", dataset["train"].shape)
print("\nFirst example:")
print(dataset["train"][0])

In [ ]:
checkpoint_path="/kaggle/input/datasets/shouldichangemyname/checkpoint-27000/checkpoint-27000"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded")
print("Vocab size:", tokenizer.vocab_size)
print("EOS Token:", tokenizer.eos_token)

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Phi-2 model loaded successfully")
print("Model device:", base_model.device)

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.1,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "dense",
        "fc1",
        "fc2"
    ]
)

peft_model = get_peft_model(
    base_model,
    lora_config
)

peft_model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/models/phi2-dcot",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    weight_decay=0.001,
    optim="paged_adamw_32bit",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    fp16=False,
    bf16=False,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

print("Training config created")

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

print("SFT trainer created")

In [ ]:
print("Resume checkpoint:", checkpoint_path)

print("\ntrainer state:")
import json

with open(
    os.path.join(checkpoint_path, "trainer_state.json"),
    "r"
) as f:
    state = json.load(f)

print("Global step:", state.get("global_step"))
print("Epoch:", state.get("epoch"))
print("Best model checkpoint:", state.get("best_model_checkpoint"))

In [ ]:
train_result = trainer.train(
    resume_from_checkpoint=checkpoint_path
)

print("Training resumed and completed.")
print(train_result)

In [ ]:
final_output = "/kaggle/working/models/phi2-dcot-final"

trainer.save_model(final_output)
tokenizer.save_pretrained(final_output)

print("Final model saved to:")
print(final_output)

In [ ]:
import os

print("Saved checkpoints:\n")

for root, dirs, files in os.walk("/kaggle/working/models/phi2-dcot"):
    if "adapter_model.safetensors" in files:
        print(root)

In [ ]:
import shutil

final_zip = "/kaggle/working/phi2-dcot-final"

shutil.make_archive(
    final_zip,
    "zip",
    root_dir="/kaggle/working/models/phi2-dcot",
    base_dir="checkpoint-31143"
)

print("Created:")
print(final_zip + ".zip")